<a href="https://colab.research.google.com/github/gauravprajapati9210/Machine-Learning-Projects/blob/main/RNN_Sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv("IMDB Dataset.csv")


In [ ]:
df.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(49582, 2)

# Applying Text-processing
#1 converting lower case

In [ ]:
df["review"]=df["review"].str.lower()



2.Removing Urls.

In [ ]:
import re
def remove_url(text):
  text = re.sub(r" http\S+" ,"",text)  # (pattern , repl,string) - eg:- https//www.google.com
  return text
df["review"] = df["review"].apply(remove_url)


Removing Punctuations

In [ ]:
def remove_punctuations(text):
  text=re.sub(r"[^A-Za-z0-9\s]","",text)
  return text
df["review"] = df["review"].apply(remove_punctuations)


# Remove html

In [ ]:
def remove_html(text):
  text=re.sub(r"<.*?>","",text)
  return text
df["review"] = df["review"].apply(remove_html)


# Removing the stop words


In [ ]:
import nltk # Natual language toolkit
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


In [ ]:
def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text = text.replace(word,"")

  return text
df["review"] = df["review"].apply(remove_stopwords)


# Stemmming

In [ ]:
# it used to take word in core form
#eg:- running --> run
 # coding -> code
 # coder --> code
from nltk.stem import PorterStemmer


In [ ]:
def stemming(text):
  ps = PorterStemmer()
  stemmed_words = []

  tokens = word_tokenize(text)
  for token in tokens:
    stemmed_token = ps.stem(token)
    stemmed_words.append(stemmed_token)
  return " ".join(stemmed_words)
df["review"]= df["review"].apply(stemming)


# Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])


In [ ]:
y = df["sentiment"]


# Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tf =TfidfVectorizer(max_features=5000)

x= tf.fit_transform(df['review'])
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057170 stored elements and shape (49582, 5000)>

# Dataset & Data laoders


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(
    x,y,test_size = 0.2,random_state=42
)


In [ ]:
import torch
from torch.utils.data import TensorDataset , DataLoader

In [ ]:
x_train = x_train.toarray()

In [ ]:
x_test = x_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set,shuffle = True,batch_size = 64)
test_loader = DataLoader(test_set,shuffle = True,batch_size = 64)

# Build our RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
  def __init__(self, input_size,hidden_size = 128,num_layers=1):
    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

    # RNN layer
    self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

    self.fc = nn.Linear(hidden_size,1)
  def forward(self,x):
    #optiojnal => shape(num of layers , batch size , hidden size)
    h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

    out, _ = self.rnn(x,h0)

    # 1st value = hidden state of all the timesteps
    # 2nd value = final hidden state of last timestep

    out = self.fc(out[:,-1,:])
    return out

In [ ]:
# model


input_size= x_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Training and Evaluation of model

In [ ]:
# Training the RNN
epochs =10
for epoch in range(epochs):
  model.train()
  for Xb , yb in train_loader:
    optimizer.zero_grad()
    Xb = Xb.unsqueeze(1) #singleton direction

    outputs = model(Xb)
    outputs = outputs.squeeze()
    outputs = torch.sigmoid(outputs)

    loss = criterion(outputs,yb)
    loss.backward() # backprop
    optimizer.step() # weights update

  print(f"epoch= {epoch+1}/{epochs} and loss = {loss.item()}")


epoch= 1/10 and loss = 0.32663264870643616
epoch= 2/10 and loss = 0.32809922099113464
epoch= 3/10 and loss = 0.39642998576164246
epoch= 4/10 and loss = 0.4547516107559204
epoch= 5/10 and loss = 0.3053019344806671
epoch= 6/10 and loss = 0.35451868176460266
epoch= 7/10 and loss = 0.1728927493095398
epoch= 8/10 and loss = 0.3214096426963806
epoch= 9/10 and loss = 0.15806324779987335
epoch= 10/10 and loss = 0.1871861070394516


Evaluation


In [ ]:
model.eval()
with torch.no_grad():
  correct_vals =0
  tot_vals =0

  for Xb,yb in test_loader:
    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)
    predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

    tot_vals += yb.size(0)
    correct_vals += (predicted == yb).sum().item()
  print(f"accuracy = {correct_vals/tot_vals * 100}")

accuracy = 85.80215791065847
